# Tutorial 17 — The Love-Hate Triangle: BSS Lattice, Bayesian Network & HLLSet Ring

Three structures are rooted in the same HLLSet ring:

```
                    HLLSet Ring
                  (ground truth)
                  XOR · AND · OR
                 ╱             ╲
              ╱    τ = P(A|B)    ╲
            ╱     (SHARED)         ╲
        BSS Lattice ←──────→ Bayesian Network
        (structural)          (epistemic)
        ORDER                 MEASURE
        ⊆, Hasse, levels     ⊥, Markov, belief
```

They **love** each other because they share the same edge-weight formula:

$$\tau(A \to B) = \frac{|A \cap B|}{|B|} = P(A \mid B)$$

They **hate** each other because they compete for interpretation:
- BSS says τ is *inclusion strength* (who contains whom?)
- BN says P(A|B) is *conditional probability* (what predicts what?)

And the ring **arbitrates** — it provides the algebraic ground truth
from which both representations are derived.

## Prerequisites

| Tutorial | Concept Used Here |
|----------|------------------|
| [01_hllset.ipynb](01_hllset.ipynb) | `HLLSet`, `union`, `intersect`, `cardinality` |
| [06_bss.ipynb](06_bss.ipynb) | `bss()`, `BSSPair`, `morphism_graph()` |
| [12_lattice_reconstruction.ipynb](12_lattice_reconstruction.ipynb) | `BSSMorphismGraph`, `LatticeReconstructor` |
| [16_bayesian.ipynb](16_bayesian.ipynb) | `prior`, `conditional`, `bayes_theorem` |

In [1]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, math
sys.path.insert(0, '..')

from core.hllset import HLLSet
from core.bitvector_ring import BitVector, BitVectorRing, verify_bridge_law
from core.bss import bss, BSSPair, bss_matrix, morphism_graph
from core.lattice_reconstruction import (
    BSSMorphismGraph, LatticeReconstructor, reconstruct_lattice,
)
from core.bayesian import prior, conditional, bayes_theorem
from core.bayesian_network import (
    HLLBayesNet,
    hllset_mutual_information,
    conditional_mutual_information,
    ring_to_bn_functor,
)

P_BITS = 10
print('Imports OK')

Imports OK


---
## Part I — The Shared Formula: τ = P(A|B)

### 1. The Identity That Creates the Triangle

Let us start with the formula that binds BSS and BN together:

$$\tau(A \to B) = \frac{|A \cap B|}{|B|} = P(A \mid B)$$

BSS computes this and calls it *inclusion*.
BN computes this and calls it *conditional probability*.
The ring provides the `intersect()` and `cardinality()` that make both possible.

In [2]:
# Create a family of overlapping HLLSets
docs = {
    'A': HLLSet.from_batch([f"token_{i}" for i in range(30)]),          # 0..29
    'B': HLLSet.from_batch([f"token_{i}" for i in range(10, 40)]),      # 10..39
    'C': HLLSet.from_batch([f"token_{i}" for i in range(25, 50)]),      # 25..49
    'D': HLLSet.from_batch([f"token_{i}" for i in range(100, 120)]),    # disjoint
}

# Show the identity: τ(A→B) = P(A|B) for every pair
print(f"{'Pair':>8}  {'BSS τ(A→B)':>12}  {'P(A|B)':>10}  {'Difference':>12}")
print("-" * 52)
for n1, h1 in docs.items():
    for n2, h2 in docs.items():
        if n1 == n2:
            continue
        # BSS τ
        tau = bss(h1, h2).tau
        # Bayesian P(A|B)
        p_a_given_b = conditional(h1, h2).value
        # They should be identical
        diff = abs(tau - p_a_given_b)
        mark = '✓' if diff < 0.001 else '≈'
        print(f"{n1}→{n2:>4}  {tau:>12.6f}  {p_a_given_b:>10.6f}  {diff:>12.6f}  {mark}")

    Pair    BSS τ(A→B)      P(A|B)    Difference
----------------------------------------------------
A→   B      0.685714    0.685714      0.000000  ✓
A→   C      0.269231    0.269231      0.000000  ✓
A→   D      0.000000    0.000000      0.000000  ✓
B→   A      0.685714    0.685714      0.000000  ✓
B→   C      0.653846    0.653846      0.000000  ✓
B→   D      0.000000    0.000000      0.000000  ✓
C→   A      0.200000    0.200000      0.000000  ✓
C→   B      0.485714    0.485714      0.000000  ✓
C→   D      0.000000    0.000000      0.000000  ✓
D→   A      0.000000    0.000000      0.000000  ✓
D→   B      0.000000    0.000000      0.000000  ✓
D→   C      0.000000    0.000000      0.000000  ✓


**The values are identical** (within floating-point rounding).
This is not coincidence — it is algebraic necessity:

$$\tau(A \to B) \equiv P(A \mid B) \equiv \frac{|A \cap B|}{|B|}$$

The ring operation `intersect()` is the common root.

---
## Part II — Three Representations of One Ring

### 2. The Ring (Ground Truth)

The HLLSet ring is a Boolean algebra with:
- **Ring addition**: XOR (symmetric difference)
- **Ring multiplication**: AND (intersection)
- **Lattice join**: OR (union)
- **Bridge law**: $A \cup B = (A \triangle B) \triangle (A \cap B)$

It contains ALL the information. Both BSS and BN are *lossy projections*.

In [3]:
# Demonstrate the ring structure at the bitvector level
N = 32
bv_a = BitVector.from_bits([0, 1, 2, 3, 4, 5, 10, 11, 12], N=N)
bv_b = BitVector.from_bits([3, 4, 5, 6, 7, 8, 10, 11], N=N)

# Ring operations
xor_ab = bv_a ^ bv_b     # Ring addition (symmetric difference)
and_ab = bv_a & bv_b     # Ring multiplication (intersection)
or_ab  = bv_a | bv_b     # Lattice join (union)

print("Ring Operations on BitVectors:")
print(f"  A          = {bv_a.to_binary_str(16)}  (popcount={bv_a.popcount()})")
print(f"  B          = {bv_b.to_binary_str(16)}  (popcount={bv_b.popcount()})")
print(f"  A ⊕ B (XOR)= {xor_ab.to_binary_str(16)}  (popcount={xor_ab.popcount()})")
print(f"  A ∧ B (AND)= {and_ab.to_binary_str(16)}  (popcount={and_ab.popcount()})")
print(f"  A ∨ B (OR) = {or_ab.to_binary_str(16)}  (popcount={or_ab.popcount()})")

# Verify bridge law: A ∪ B = (A △ B) △ (A ∩ B)
bridge_ok = verify_bridge_law(bv_a, bv_b)
print(f"\n  Bridge law: A ∪ B = (A △ B) △ (A ∩ B)  →  {'✓ Verified' if bridge_ok else '✗ Failed'}")

# Ring invariants
print(f"  X ⊕ X = 0: {(bv_a ^ bv_a).is_zero()}  (characteristic 2)")
print(f"  X ∧ X = X: {(bv_a & bv_a) == bv_a}  (idempotent)")

Ring Operations on BitVectors:
  A          = 0001110000111111  (popcount=9)
  B          = 0000110111111000  (popcount=8)
  A ⊕ B (XOR)= 0001000111000111  (popcount=7)
  A ∧ B (AND)= 0000110000111000  (popcount=5)
  A ∨ B (OR) = 0001110111111111  (popcount=12)

  Bridge law: A ∪ B = (A △ B) △ (A ∩ B)  →  ✓ Verified
  X ⊕ X = 0: True  (characteristic 2)
  X ∧ X = X: True  (idempotent)


### 3. BSS Lattice (Order Representation)

The BSS lattice **projects** the ring onto a partial order.

What it reveals:
- Subset relations ($A \subseteq B$)
- Covering relations (Hasse diagram)
- Levels and antichains
- Morphism strength $(\tau, \rho)$

What it loses:
- XOR (ring addition)
- The bridge law

In [4]:
# Build BSS lattice from our document family
hll_list = list(docs.values())
labels = list(docs.keys())

# Reconstruct lattice
lattice_result = reconstruct_lattice(
    hll_list, node_ids=labels,
    tau_threshold=0.5,   # Lower threshold to see more structure
    subset_tau=0.85,
)

print("BSS LATTICE VIEW")
print("=" * 50)
print(f"  Nodes: {lattice_result.num_nodes}")
print(f"  Edges: {lattice_result.num_edges}")
print(f"  Hasse edges: {lattice_result.num_hasse_edges}")
print(f"  Levels: {lattice_result.num_levels}")
print(f"  Height: {lattice_result.height}")
print(f"  Width: {lattice_result.width}")

print("\n  Edge details (BSS interpretation):")
for edge in lattice_result.edges[:8]:
    print(f"    {edge.source_id}→{edge.target_id}: "
          f"τ={edge.tau_forward:.4f}, ρ={edge.rho_forward:.4f}, "
          f"type={edge.edge_type}")

BSS LATTICE VIEW
  Nodes: 4
  Edges: 4
  Hasse edges: 0
  Levels: 1
  Height: 0
  Width: 4

  Edge details (BSS interpretation):
    A→B: τ=0.6857, ρ=0.3429, type=overlap
    B→A: τ=0.6857, ρ=0.3143, type=overlap
    B→C: τ=0.6538, ρ=0.6923, type=overlap
    C→B: τ=0.4857, ρ=0.2571, type=overlap


### 4. Bayesian Network (Measure Representation)

The BN **projects** the same ring onto a probability space.

What it reveals:
- Conditional probabilities $P(A|B)$
- Independence structure (d-separation)
- Markov blankets
- Causal ordering

What it loses:
- XOR (ring addition)
- ρ (exclusion metric)
- Subset direction (partial order)

In [5]:
# Build Bayesian Network from the same HLLSets
bn = HLLBayesNet.from_hllsets(docs, threshold=0.1)

print("BAYESIAN NETWORK VIEW")
print("=" * 50)
print(f"  Nodes: {bn.num_nodes}")
print(f"  Edges: {bn.num_edges}")

print("\n  CPT details (BN interpretation):")
for node_id in bn.node_ids:
    entries = bn.get_cpt(node_id)
    for e in entries:
        print(f"    P({e.child_id}|{e.parent_id}) = {e.probability:.4f}  "
              f"[BSS: τ={e.bss_tau:.4f}, ρ={e.bss_rho:.4f}]")

print("\n  Markov blankets:")
for nid in bn.node_ids:
    mb = bn.markov_blanket(nid)
    print(f"    {nid}: {mb}")

BAYESIAN NETWORK VIEW
  Nodes: 4
  Edges: 3

  CPT details (BN interpretation):
    P(A|B) = 0.6857  [BSS: τ=0.6857, ρ=0.3429]
    P(C|B) = 0.4857  [BSS: τ=0.4857, ρ=0.2571]
    P(C|A) = 0.2000  [BSS: τ=0.2000, ρ=0.5714]

  Markov blankets:
    A: MarkovBlanket(A: 1 parents, 1 children, 1 co-parents, total=2)
    B: MarkovBlanket(B: 0 parents, 2 children, 1 co-parents, total=2)
    C: MarkovBlanket(C: 2 parents, 0 children, 0 co-parents, total=2)
    D: MarkovBlanket(D: 0 parents, 0 children, 0 co-parents, total=0)


---
## Part III — Proving the Graph Isomorphism

### 5. BSS Lattice ≅ BN (as weighted graphs)

We claim these two representations produce **isomorphic weighted graphs**:
- Same nodes
- Same edges
- Same edge weights: $\tau = P(A|B)$

The `isomorphism_witness()` function formally verifies this.

In [6]:
# Build BSS morphism graph
bss_graph = BSSMorphismGraph(tau_threshold=0.1, rho_threshold=0.9, subset_tau=0.85)
for nid, hll in docs.items():
    bss_graph.add_node(nid, hll)
bss_graph.compute_all_edges()

# Build BN from the same BSS graph (the isomorphism constructor)
bn_from_bss = HLLBayesNet.from_bss_graph(bss_graph)

# Verify isomorphism
witness = bn_from_bss.isomorphism_witness(bss_graph, tolerance=0.01)

print("ISOMORPHISM VERIFICATION")
print("=" * 60)
print(f"  {witness}")
print(f"  Max weight error: {witness.max_weight_error:.8f}")
print(f"  Mean weight error: {witness.mean_weight_error:.8f}")
print()

# Show some edge correspondences
print("  Edge-by-edge correspondence:")
for corr in witness.edge_correspondences[:6]:
    print(f"    {corr['source']}→{corr['target']}: "
          f"BSS τ={corr['bss_tau']:.6f} = BN P={corr['bn_probability']:.6f} "
          f"(err={corr['error']:.8f}) {'✓' if corr['match'] else '✗'}")

print(f"\n  {witness.structural_note[:200]}...")

ISOMORPHISM VERIFICATION
  Isomorphism(BSS ≅ BN: 4 nodes, 6 edges, max_err=0.000000)
  Max weight error: 0.00000000
  Mean weight error: 0.00000000

  Edge-by-edge correspondence:
    A→B: BSS τ=0.685714 = BN P=0.685714 (err=0.00000000) ✓
    B→A: BSS τ=0.685714 = BN P=0.685714 (err=0.00000000) ✓
    A→C: BSS τ=0.269231 = BN P=0.269231 (err=0.00000000) ✓
    C→A: BSS τ=0.200000 = BN P=0.200000 (err=0.00000000) ✓
    B→C: BSS τ=0.653846 = BN P=0.653846 (err=0.00000000) ✓
    C→B: BSS τ=0.485714 = BN P=0.485714 (err=0.00000000) ✓

  GRAPH ISOMORPHISM VERIFIED. BSS τ(A→B) = P(A|B) for all edges — both representations project the same HLLSet ring onto the same weighted graph. The BSS lattice carries additional ρ (exclusion) informa...


In [7]:
# ── Deeper Analysis: The ρ Subgraph (BSS-only structure) ─────────────
print("ρ-SUBGRAPH: THE EXCLUSION NETWORK (BSS-only)")
print("=" * 60)
print("""
While τ = P(A|B) is shared by both BSS and BN,
ρ = |A \\ B| / |B| captures what BSS sees but BN ignores:
  
  τ(A→B) measures:  "How much of B is covered by A?"
  ρ(A→B) measures:  "How much of B is OUTSIDE of A?"
  
  τ + ρ ≤ 1  (the morphism inequality — BSS law, not BN law)
  
The ρ-subgraph shows DIVERGENCE rather than overlap.
High ρ means "these sets are going different directions."
""")

# Build the ρ-subgraph: edges where ρ is significant
rho_threshold = 0.3  # Edges with ρ > 0.3 show significant exclusion

print(f"ρ-Subgraph (ρ > {rho_threshold}):")
print("-" * 50)
print(f"{'Edge':>10}  {'τ (overlap)':>12}  {'ρ (exclusion)':>14}  {'τ+ρ':>8}  {'Interpretation'}")
print("-" * 70)

rho_edges = []
for n1, h1 in docs.items():
    for n2, h2 in docs.items():
        if n1 == n2:
            continue
        pair = bss(h1, h2)
        if pair.rho > rho_threshold:
            rho_edges.append((n1, n2, pair.tau, pair.rho))
            # Interpretation
            if pair.rho > 0.8:
                interp = "Almost disjoint"
            elif pair.rho > 0.5:
                interp = "Mostly divergent"
            else:
                interp = "Partial exclusion"
            print(f"{n1}→{n2:>4}  {pair.tau:>12.4f}  {pair.rho:>14.4f}  {pair.tau + pair.rho:>8.4f}  {interp}")

print(f"\nρ-subgraph has {len(rho_edges)} edges (vs {bn.num_edges} in full graph)")

# Compare: τ-subgraph (high overlap) vs ρ-subgraph (high exclusion)
print("\n" + "=" * 60)
print("DUAL SUBGRAPHS: τ vs ρ")
print("=" * 60)

tau_threshold = 0.3
tau_edges = []
for n1, h1 in docs.items():
    for n2, h2 in docs.items():
        if n1 == n2:
            continue
        pair = bss(h1, h2)
        if pair.tau > tau_threshold:
            tau_edges.append((n1, n2, pair.tau, pair.rho))

print(f"""
  τ-subgraph (τ > {tau_threshold}): {len(tau_edges)} edges — "Who overlaps with whom?"
  ρ-subgraph (ρ > {rho_threshold}): {len(rho_edges)} edges — "Who excludes whom?"
  
  These are COMPLEMENTARY views:
    • High τ, low ρ  →  Strong containment (A ⊆ B approximately)
    • Low τ, high ρ  →  Strong divergence (A and B are "going different ways")
    • Both moderate   →  Partial overlap, partial divergence
    
  BN sees ONLY the τ-subgraph (as P(A|B)).
  BSS sees BOTH — the full morphism structure.
""")

# The ρ-subgraph reveals structure BN cannot see
print("WHAT ρ REVEALS THAT BN CANNOT SEE:")
print("-" * 50)
for n1, n2, tau, rho in rho_edges:
    bn_prob = conditional(docs[n1], docs[n2]).value  # = τ
    print(f"  {n1}→{n2}: BN sees P={bn_prob:.4f}, "
          f"BSS adds ρ={rho:.4f} (exclusion)")
    print(f"          → {100*rho:.1f}% of {n2} is OUTSIDE {n1}")

print(f"""
\nThe ρ-subgraph is the "dark matter" of the BN:
  • Same nodes, different edges
  • Captures what WON'T transfer, not what WILL
  • Essential for understanding divergence and novelty
""")

ρ-SUBGRAPH: THE EXCLUSION NETWORK (BSS-only)

While τ = P(A|B) is shared by both BSS and BN,
ρ = |A \ B| / |B| captures what BSS sees but BN ignores:

  τ(A→B) measures:  "How much of B is covered by A?"
  ρ(A→B) measures:  "How much of B is OUTSIDE of A?"

  τ + ρ ≤ 1  (the morphism inequality — BSS law, not BN law)

The ρ-subgraph shows DIVERGENCE rather than overlap.
High ρ means "these sets are going different directions."

ρ-Subgraph (ρ > 0.3):
--------------------------------------------------
      Edge   τ (overlap)   ρ (exclusion)       τ+ρ  Interpretation
----------------------------------------------------------------------
A→   B        0.6857          0.3429    1.0286  Partial exclusion
A→   C        0.2692          1.0000    1.2692  Almost disjoint
A→   D        0.0000          1.0000    1.0000  Almost disjoint
B→   A        0.6857          0.3143    1.0000  Partial exclusion
B→   C        0.6538          0.6923    1.3462  Mostly divergent
B→   D        0.0000          1.

In [8]:
# Add after the ρ-subgraph analysis cell

# ── BN Benefit: Controlled Pruning by Relevance ──────────────────────
print("BN BENEFIT: CONTROLLED PRUNING BY RELEVANCE")
print("=" * 60)
print("""
Standard BN pruning uses only P(A|B) — but this misses crucial info:
  • High P(A|B) with high ρ → A covers B, but B has MUCH MORE content
  • High P(A|B) with low ρ  → A covers B, and B is mostly contained in A

BSS ρ enables RELEVANCE-AWARE pruning:
  • Keep nodes with high τ AND low ρ (strong, focused relationships)
  • Prune nodes with high ρ (too divergent to be useful parents)
""")

# Define relevance score: high τ, low ρ = highly relevant
def relevance_score(tau: float, rho: float) -> float:
    """
    Relevance = τ * (1 - ρ)
    - τ alone: how much of target is covered by source
    - (1 - ρ): how focused the relationship is (low divergence)
    - Product: relevant if covers well AND doesn't diverge much
    """
    return tau * (1 - rho)

print("\nRELEVANCE SCORES (τ * (1-ρ)):")
print("-" * 70)
print(f"{'Edge':>10}  {'τ':>8}  {'ρ':>8}  {'Relevance':>12}  {'Standard BN':>14}  {'Action'}")
print("-" * 70)

edge_data = []
for n1, h1 in docs.items():
    for n2, h2 in docs.items():
        if n1 == n2:
            continue
        pair = bss(h1, h2)
        rel = relevance_score(pair.tau, pair.rho)
        edge_data.append((n1, n2, pair.tau, pair.rho, rel))

# Sort by relevance
edge_data.sort(key=lambda x: x[4], reverse=True)

for n1, n2, tau, rho, rel in edge_data:
    # Standard BN would use τ alone
    bn_decision = "KEEP" if tau > 0.3 else "PRUNE"
    # BSS-informed decision uses relevance
    if rel > 0.3:
        action = "✓ KEEP (relevant)"
    elif rel > 0.1:
        action = "? REVIEW"
    else:
        action = "✗ PRUNE (divergent)"
    
    print(f"{n1}→{n2:>4}  {tau:>8.4f}  {rho:>8.4f}  {rel:>12.4f}  {bn_decision:>14}  {action}")

# Demonstrate pruning improvement
print("\n" + "=" * 60)
print("PRUNING COMPARISON")
print("=" * 60)

tau_threshold = 0.3
relevance_threshold = 0.15

bn_kept = [e for e in edge_data if e[2] > tau_threshold]  # τ > threshold
bss_kept = [e for e in edge_data if e[4] > relevance_threshold]  # relevance > threshold

print(f"""
  Standard BN (τ > {tau_threshold}):
    Keeps {len(bn_kept)} edges: {[(e[0], e[1]) for e in bn_kept]}
    
  BSS-informed (relevance > {relevance_threshold}):
    Keeps {len(bss_kept)} edges: {[(e[0], e[1]) for e in bss_kept]}
""")

# Find edges where BN and BSS disagree
bn_only = [e for e in bn_kept if e not in bss_kept]
bss_only = [e for e in bss_kept if e not in bn_kept]

if bn_only:
    print("  Edges BN keeps but BSS prunes (HIGH DIVERGENCE):")
    for e in bn_only:
        print(f"    {e[0]}→{e[1]}: τ={e[2]:.4f} looks good, but ρ={e[3]:.4f} → {100*e[3]:.0f}% divergent!")

if bss_only:
    print("\n  Edges BSS keeps but BN misses:")
    for e in bss_only:
        print(f"    {e[0]}→{e[1]}: τ={e[2]:.4f} below BN threshold, but relevance={e[4]:.4f}")

# Practical application: parent selection
print("\n" + "=" * 60)
print("PRACTICAL APPLICATION: OPTIMAL PARENT SELECTION")
print("=" * 60)

for target in docs.keys():
    print(f"\n  Best parents for '{target}':")
    candidates = [(n, t, r, rel) for n, tgt, t, r, rel in edge_data if tgt == target]
    candidates.sort(key=lambda x: x[3], reverse=True)  # Sort by relevance
    
    for parent, tau, rho, rel in candidates[:3]:
        quality = "★★★" if rel > 0.3 else ("★★" if rel > 0.15 else "★")
        print(f"    {parent}: τ={tau:.4f}, ρ={rho:.4f}, relevance={rel:.4f} {quality}")
    
    if candidates:
        best = candidates[0]
        print(f"    → Recommended: {best[0]} (highest relevance={best[3]:.4f})")

print(f"""
\n{'='*60}
KEY INSIGHT: ρ-INFORMED PRUNING
{'='*60}

Standard BN pruning (τ only):
  • "Does A predict B?" — Yes if τ > threshold
  • BLIND to divergence — keeps noisy, unfocused edges

BSS-informed pruning (τ and ρ):
  • "Does A RELEVANTLY predict B?" — Yes if τ high AND ρ low
  • AWARE of divergence — prunes edges where source/target go different ways

Benefits:
  1. Smaller, cleaner BN (fewer spurious edges)
  2. Better inference (less noise propagation)  
  3. Interpretable pruning (explain WHY an edge was removed)
  4. Tunable relevance threshold (trade precision vs recall)
""")

BN BENEFIT: CONTROLLED PRUNING BY RELEVANCE

Standard BN pruning uses only P(A|B) — but this misses crucial info:
  • High P(A|B) with high ρ → A covers B, but B has MUCH MORE content
  • High P(A|B) with low ρ  → A covers B, and B is mostly contained in A

BSS ρ enables RELEVANCE-AWARE pruning:
  • Keep nodes with high τ AND low ρ (strong, focused relationships)
  • Prune nodes with high ρ (too divergent to be useful parents)


RELEVANCE SCORES (τ * (1-ρ)):
----------------------------------------------------------------------
      Edge         τ         ρ     Relevance     Standard BN  Action
----------------------------------------------------------------------
B→   A    0.6857    0.3143        0.4702            KEEP  ✓ KEEP (relevant)
A→   B    0.6857    0.3429        0.4506            KEEP  ✓ KEEP (relevant)
C→   B    0.4857    0.2571        0.3608            KEEP  ✓ KEEP (relevant)
B→   C    0.6538    0.6923        0.2012            KEEP  ? REVIEW
C→   A    0.2000    0.5714     

### 6. Why Isomorphic as Graphs but NOT as Algebras

The graph isomorphism is trivially true (same formula). But the two
representations carry **different additional structure**:

| Property | BSS Lattice | Bayesian Network | Shared? |
|----------|-------------|-----------------|--------|
| Node set | HLLSets | HLLSets | ✓ Same |
| Edge weight | τ = \|A∩B\|/\|B\| | P(A\|B) = \|A∩B\|/\|B\| | ✓ Same |
| ρ (exclusion) | \|A\\B\|/\|B\| | — | ✗ BSS only |
| τ + ρ ≤ 1 | Morphism law | — | ✗ BSS only |
| Bayes' theorem | — | P(A\|B)P(B) = P(B\|A)P(A) | ✗ BN only |
| d-separation | — | A ⊥ B \| C | ✗ BN only |
| Markov blanket | — | Parents ∪ children ∪ co-parents | ✗ BN only |
| Partial order | ⊆ from τ ≈ 1 | — | ✗ BSS only |
| Hasse diagram | Transitive reduction | — | ✗ BSS only |
| Belief propagation | — | Message passing | ✗ BN only |

The relationship is a **forgetful functor**: both project the ring
onto the same graph skeleton, but each "forgets" different aspects
and adds different algebraic operations on top.

---
## Part IV — The BN Toolbox on HLLSets

### 7. Conditional Independence (d-Separation)

In standard BN, d-separation uses the graph structure to determine
conditional independence. In HLLSet BN, we can test it **directly**
using set operations:

$$A \perp B \mid C \iff P(A \cap B \mid C) \approx P(A \mid C) \cdot P(B \mid C)$$

This is the set-theoretic definition — no graph needed!

In [9]:
# Test conditional independence
print("CONDITIONAL INDEPENDENCE TESTS")
print("=" * 60)

# A and D should be independent (disjoint sets)
result = bn.conditional_independence('A', 'D')
print(f"  {result}")
print(f"    {result.explanation}")
print()

# A and B should be dependent (overlapping)
result = bn.conditional_independence('A', 'B')
print(f"  {result}")
print(f"    {result.explanation}")
print()

# A and C conditioned on B — does B screen off C from A?
result = bn.conditional_independence('A', 'C', given_ids=['B'])
print(f"  {result}")
print(f"    {result.explanation}")

CONDITIONAL INDEPENDENCE TESTS
  Independence(A ⊬⊥ D | ∅, I=0.5247)
    A ⊬⊥ D | {∅}: P(A∩B|C)=0.0000 ≠ P(A|C)·P(B|C)=0.1431 (deviation=0.1431 ≥ tol=0.05)

  Independence(A ⊬⊥ B | ∅, I=0.0000)
    A ⊬⊥ B | {∅}: P(A∩B|C)=0.3200 ≠ P(A|C)·P(B|C)=0.2178 (deviation=0.1022 ≥ tol=0.05)

  Independence(A ⊬⊥ C | B, I=0.3355)
    A ⊬⊥ C | {B}: P(A∩B|C)=0.2000 ≠ P(A|C)·P(B|C)=0.3331 (deviation=0.1331 ≥ tol=0.05)


### 8. Mutual Information

$$I(A; B) = H(A) + H(B) - H(A, B)$$

Mutual information measures the *shared information* between two
HLLSets. It is zero iff they are independent.

**BSS has no analogue** — τ and ρ measure directional overlap,
not symmetric information content.

In [10]:
U = HLLSet.merge(list(docs.values()))

print("MUTUAL INFORMATION (BN-only concept)")
print("=" * 50)
for n1, h1 in docs.items():
    for n2, h2 in docs.items():
        if n1 >= n2:
            continue
        mi = hllset_mutual_information(h1, h2, U)
        tau_fwd = bss(h1, h2).tau
        tau_bwd = bss(h2, h1).tau
        print(f"  I({n1};{n2}) = {mi:.4f} bits  "
              f"[BSS: τ({n1}→{n2})={tau_fwd:.4f}, τ({n2}→{n1})={tau_bwd:.4f}]")

print("\n  Note: MI is symmetric; BSS τ is NOT.")
print("  MI captures 'shared info'; τ captures 'directional coverage'.")

MUTUAL INFORMATION (BN-only concept)
  I(A;B) = 0.1251 bits  [BSS: τ(A→B)=0.6857, τ(B→A)=0.6857]
  I(A;C) = 0.0618 bits  [BSS: τ(A→C)=0.2692, τ(C→A)=0.2000]
  I(A;D) = 0.3646 bits  [BSS: τ(A→D)=0.0000, τ(D→A)=0.0000]
  I(B;C) = 0.0544 bits  [BSS: τ(B→C)=0.6538, τ(C→B)=0.4857]
  I(B;D) = 0.3646 bits  [BSS: τ(B→D)=0.0000, τ(D→B)=0.0000]
  I(C;D) = 0.2377 bits  [BSS: τ(C→D)=0.0000, τ(D→C)=0.0000]

  Note: MI is symmetric; BSS τ is NOT.
  MI captures 'shared info'; τ captures 'directional coverage'.


### 9. Belief Propagation

When we observe evidence (a specific HLLSet), beliefs across the
network update. This is standard BN inference — but here the
"messages" are computed from `intersect()` and `cardinality()`.

BSS has no mechanism for this: observing one node doesn't "propagate"
through the lattice in any standard way.

In [11]:
# Belief propagation: no evidence vs evidence on B
print("BELIEF PROPAGATION")
print("=" * 50)

# Prior beliefs (no evidence)
prior_beliefs = bn.belief_propagation()
print("  Prior beliefs (no evidence):")
for nid, prob in prior_beliefs.probabilities.items():
    print(f"    P({nid}) = {prob:.4f}")

# Posterior beliefs (evidence: we observe B)
posterior_beliefs = bn.belief_propagation(evidence={'B': docs['B']})
print("\n  Posterior beliefs (evidence = B):")
for nid, prob in posterior_beliefs.probabilities.items():
    delta = prob - prior_beliefs.probabilities[nid]
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '→')
    print(f"    P({nid}|B) = {prob:.4f}  (Δ={delta:+.4f} {arrow})")

print("\n  BSS lattice cannot do this — it has no 'evidence' mechanism.")

BELIEF PROPAGATION
  Prior beliefs (no evidence):
    P(A) = 0.6857
    P(B) = 0.4667
    P(C) = 0.3157
    P(D) = 0.3067

  Posterior beliefs (evidence = B):
    P(A|B) = 0.6857  (Δ=+0.0000 →)
    P(B|B) = 1.0000  (Δ=+0.5333 ↑)
    P(C|B) = 0.3695  (Δ=+0.0538 ↑)
    P(D|B) = 0.0000  (Δ=-0.3067 ↓)

  BSS lattice cannot do this — it has no 'evidence' mechanism.


### 10. Structure Learning = Lattice Reconstruction

Here is the deepest connection: **BN structure learning IS lattice
reconstruction, just with different vocabulary.**

| BN Structure Learning | Lattice Reconstruction | Same Code? |
|----------------------|----------------------|------------|
| Compute pairwise P(A\|B) | Compute pairwise BSS τ | ✓ Same formula |
| Threshold edges | Threshold morphisms | ✓ Same logic |
| Orient edges (causal) | Orient by subset (⊆) | Similar |
| Learn DAG skeleton | Compute Hasse diagram | Similar |
| Score: BIC, MDL | Score: morphism strength | Different |

The `LatticeReconstructor` is, in BN language, a **constraint-based
structure learning algorithm** using conditional independence tests.

In [12]:
# Side-by-side: lattice reconstruction vs BN structure learning
print("LATTICE RECONSTRUCTION vs BN STRUCTURE LEARNING")
print("=" * 60)

# Same input, two interpretations
print(f"\n  Input: {len(docs)} HLLSets")
print(f"  BSS Lattice: {lattice_result.num_edges} edges, "
      f"{lattice_result.num_hasse_edges} Hasse, "
      f"{lattice_result.num_levels} levels")
print(f"  BN Network:  {bn.num_edges} edges, "
      f"{len([nid for nid in bn.node_ids if not bn.parents(nid)])} roots, "
      f"{len([nid for nid in bn.node_ids if not bn.children(nid)])} leaves")

print("\n  Both learned from the SAME pairwise computation:")
print("    BSS calls it 'morphism graph construction'")
print("    BN  calls it 'structure learning'")
print("    Ring calls it 'pairwise intersect + cardinality'")

LATTICE RECONSTRUCTION vs BN STRUCTURE LEARNING

  Input: 4 HLLSets
  BSS Lattice: 4 edges, 0 Hasse, 1 levels
  BN Network:  3 edges, 2 roots, 2 leaves

  Both learned from the SAME pairwise computation:
    BSS calls it 'morphism graph construction'
    BN  calls it 'structure learning'
    Ring calls it 'pairwise intersect + cardinality'


### 10.1. BN Pays Back: Optimizing BSS Lattice via Structure Learning 

In [15]:
# Add after the "LATTICE RECONSTRUCTION vs BN STRUCTURE LEARNING" cell

# ── BN Pays Back: Optimizing BSS Lattice via Structure Learning ─────
print("BN PAYS BACK: OPTIMIZING BSS LATTICE RECONSTRUCTION")
print("=" * 70)
print("""
BSS lattice reconstruction uses fixed thresholds (τ_threshold, ρ_threshold).
BN structure learning offers ADAPTIVE techniques:

  1. Information-theoretic thresholding (MI > threshold)
  2. Conditional independence pruning (remove redundant edges)
  3. Score-based optimization (BIC, MDL)
  4. Markov blanket sparsification

These techniques can IMPROVE the BSS lattice by:
  • Removing spurious edges (noise reduction)
  • Identifying optimal τ thresholds (data-driven)
  • Detecting conditional redundancy (A→C redundant given A→B→C)
""")

# ── Technique 1: MI-based Threshold Selection ────────────────────────
print("\n" + "=" * 70)
print("TECHNIQUE 1: MUTUAL INFORMATION THRESHOLD")
print("=" * 70)

U = HLLSet.merge(list(docs.values()))

# Compute MI for all pairs and find natural threshold
mi_data = []
for n1, h1 in docs.items():
    for n2, h2 in docs.items():
        if n1 >= n2:
            continue
        mi = hllset_mutual_information(h1, h2, U)
        tau_fwd = bss(h1, h2).tau
        tau_bwd = bss(h2, h1).tau
        mi_data.append((n1, n2, mi, tau_fwd, tau_bwd))

# Sort by MI
mi_data.sort(key=lambda x: x[2], reverse=True)

print("\nMI-ranked edges (descending):")
print(f"{'Pair':>8}  {'MI (bits)':>12}  {'τ(A→B)':>10}  {'τ(B→A)':>10}  {'BSS keeps?':>12}")
print("-" * 60)

# Find the "elbow" — largest MI gap
mi_values = [m[2] for m in mi_data]
mi_gaps = [mi_values[i] - mi_values[i+1] for i in range(len(mi_values)-1)]
if mi_gaps:
    elbow_idx = mi_gaps.index(max(mi_gaps))
    adaptive_mi_threshold = (mi_values[elbow_idx] + mi_values[elbow_idx+1]) / 2
else:
    adaptive_mi_threshold = 0.1

for n1, n2, mi, tau_fwd, tau_bwd in mi_data:
    bss_keeps = "✓" if max(tau_fwd, tau_bwd) > 0.3 else "✗"
    mi_keeps = "★" if mi > adaptive_mi_threshold else ""
    print(f"{n1}↔{n2:>4}  {mi:>12.4f}  {tau_fwd:>10.4f}  {tau_bwd:>10.4f}  {bss_keeps:>12} {mi_keeps}")

print(f"\n  Adaptive MI threshold (elbow method): {adaptive_mi_threshold:.4f} bits")
print(f"  → Suggests keeping edges with MI > {adaptive_mi_threshold:.4f}")
print(f"  → This translates to τ threshold ≈ {min(t for _, _, mi, t, _ in mi_data if mi > adaptive_mi_threshold):.4f}")

# ── Technique 2: Conditional Independence Pruning ────────────────────
print("\n" + "=" * 70)
print("TECHNIQUE 2: CONDITIONAL INDEPENDENCE PRUNING")
print("=" * 70)
print("""
BSS keeps all edges with τ > threshold.
BN asks: "Is this edge REDUNDANT given other edges?"

If A ⊥ C | B, then A→C is redundant (B mediates the relationship).
We can PRUNE A→C from the lattice without losing information.
""")

# Check for redundant edges via conditional independence
print("\nRedundancy analysis:")
print("-" * 60)

redundant_edges = []
for n1, h1 in docs.items():
    for n3, h3 in docs.items():
        if n1 == n3:
            continue
        # Check if any intermediate node screens off n1 from n3
        for n2, h2 in docs.items():
            if n2 in (n1, n3):
                continue
            # Test: is n1 ⊥ n3 | n2?
            result = bn.conditional_independence(n1, n3, given_ids=[n2])
            if result.independent:
                tau_direct = bss(h1, h3).tau
                tau_via = bss(h1, h2).tau * bss(h2, h3).tau  # Approximate path strength
                redundant_edges.append((n1, n3, n2, tau_direct, tau_via))
                print(f"  {n1}→{n3} is REDUNDANT given {n2}")
                print(f"    Direct τ({n1}→{n3}) = {tau_direct:.4f}")
                print(f"    Path τ({n1}→{n2})·τ({n2}→{n3}) ≈ {tau_via:.4f}")
                print(f"    → Can prune {n1}→{n3} from lattice!")

if not redundant_edges:
    print("  No redundant edges found (all edges carry unique information)")

# ── Technique 3: Score-based Optimization ────────────────────────────
print("\n" + "=" * 70)
print("TECHNIQUE 3: SCORE-BASED τ OPTIMIZATION")
print("=" * 70)
print("""
BN uses scores like BIC (Bayesian Information Criterion):
  BIC = log P(data | structure) - (k/2) log(n)

For BSS lattice, we can define an analogous score:
  BSS_score = Σ τ(edges) - λ · |edges|

This balances coverage (high τ) vs complexity (few edges).
""")

def bss_score(edges: list, lambda_penalty: float = 0.1) -> float:
    """Score a set of edges: sum of τ minus complexity penalty."""
    tau_sum = sum(e[2] for e in edges)  # e = (src, tgt, tau, rho)
    return tau_sum - lambda_penalty * len(edges)

# Collect all possible edges with their τ values
all_edges = []
for n1, h1 in docs.items():
    for n2, h2 in docs.items():
        if n1 == n2:
            continue
        pair = bss(h1, h2)
        all_edges.append((n1, n2, pair.tau, pair.rho))

# Try different τ thresholds and score them
print("\nτ threshold optimization:")
print(f"{'τ threshold':>12}  {'# edges':>10}  {'Σ τ':>10}  {'BSS score':>12}  {'Optimal?'}")
print("-" * 60)

best_threshold = 0.0
best_score = float('-inf')
lambda_penalty = 0.15

for tau_thresh in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    kept_edges = [e for e in all_edges if e[2] > tau_thresh]
    score = bss_score(kept_edges, lambda_penalty)
    is_best = score > best_score
    if is_best:
        best_score = score
        best_threshold = tau_thresh
    print(f"{tau_thresh:>12.1f}  {len(kept_edges):>10}  "
          f"{sum(e[2] for e in kept_edges):>10.2f}  {score:>12.4f}  "
          f"{'★ BEST' if is_best else ''}")

print(f"\n  Optimal τ threshold: {best_threshold:.1f}")
print(f"  → BN scoring suggests BSS should use τ_threshold = {best_threshold:.1f}")

# ── Technique 4: Markov Blanket Sparsification ───────────────────────
print("\n" + "=" * 70)
print("TECHNIQUE 4: MARKOV BLANKET SPARSIFICATION")
print("=" * 70)
print("""
The Markov blanket of a node contains ALL nodes needed to predict it.
Any edge OUTSIDE the Markov blanket is redundant for that node.

For BSS lattice: keep only edges within Markov blankets.
""")

print("\nMarkov blanket analysis:")
essential_edges = set()
for nid in bn.node_ids:
    mb = bn.markov_blanket(nid)
    print(f"  MB({nid}) = {mb}")
    # Edges within Markov blanket are essential
    # Try common attribute names for the node set
    if hasattr(mb, 'members'):
        mb_nodes = mb.members
    elif hasattr(mb, 'nodes'):
        mb_nodes = mb.nodes
    elif hasattr(mb, 'node_ids'):
        mb_nodes = mb.node_ids
    else:
        # Fallback: try to get all related nodes
        mb_nodes = set()
        if hasattr(mb, 'parents'):
            mb_nodes |= set(mb.parents)
        if hasattr(mb, 'children'):
            mb_nodes |= set(mb.children)
        if hasattr(mb, 'spouses'):
            mb_nodes |= set(mb.spouses)
    
    for other in mb_nodes:
        essential_edges.add((nid, other))
        essential_edges.add((other, nid))

all_edge_pairs = {(e[0], e[1]) for e in all_edges if e[2] > 0.1}
prunable = all_edge_pairs - essential_edges

print(f"\n  Total edges (τ > 0.1): {len(all_edge_pairs)}")
print(f"  Essential (in Markov blankets): {len(essential_edges)}")
print(f"  Prunable (outside blankets): {len(prunable)}")

if prunable:
    print(f"\n  Prunable edges: {prunable}")
    print("  → These edges can be removed from BSS lattice!")

# ── Summary: BN → BSS Optimization ───────────────────────────────────
print("\n" + "=" * 70)
print("SUMMARY: BN OPTIMIZATIONS FOR BSS LATTICE")
print("=" * 70)
print(f"""
  ┌─────────────────────────────────────────────────────────────────┐
  │ BN Technique              │ BSS Lattice Improvement             │
  ├───────────────────────────┼─────────────────────────────────────┤
  │ MI thresholding           │ Adaptive τ = {adaptive_mi_threshold:.2f} (data-driven)   │
  │ Conditional independence  │ Prune {len(redundant_edges)} redundant edges              │
  │ Score optimization        │ Optimal τ_threshold = {best_threshold:.1f}              │
  │ Markov blanket            │ Keep only {len(essential_edges)//2} essential edge pairs     │
  └─────────────────────────────────────────────────────────────────┘

  The love-hate triangle becomes COLLABORATIVE:
    • BSS provides ρ for better BN pruning
    • BN provides structure learning for better BSS lattices
    • Ring provides the ground truth for both
""")

BN PAYS BACK: OPTIMIZING BSS LATTICE RECONSTRUCTION

BSS lattice reconstruction uses fixed thresholds (τ_threshold, ρ_threshold).
BN structure learning offers ADAPTIVE techniques:

  1. Information-theoretic thresholding (MI > threshold)
  2. Conditional independence pruning (remove redundant edges)
  3. Score-based optimization (BIC, MDL)
  4. Markov blanket sparsification

These techniques can IMPROVE the BSS lattice by:
  • Removing spurious edges (noise reduction)
  • Identifying optimal τ thresholds (data-driven)
  • Detecting conditional redundancy (A→C redundant given A→B→C)


TECHNIQUE 1: MUTUAL INFORMATION THRESHOLD

MI-ranked edges (descending):
    Pair     MI (bits)      τ(A→B)      τ(B→A)    BSS keeps?
------------------------------------------------------------
A↔   D        0.3646      0.0000      0.0000             ✗ ★
B↔   D        0.3646      0.0000      0.0000             ✗ ★
C↔   D        0.2377      0.0000      0.0000             ✗ 
A↔   B        0.1251      0.6857

---
## Part V — The Forgetful Functor

### 11. What Each Representation Forgets

The `ring_to_bn_functor()` documents precisely what is preserved
and what is lost when projecting the ring into the BN view.

In [16]:
# The forgetful functor: Ring → BN
functor = ring_to_bn_functor(docs, threshold=0.1)

print("FORGETFUL FUNCTOR: Ring → BN")
print("=" * 60)

print("\n  PRESERVED (in both BSS and BN):")
print(f"    Nodes: {functor['preserved']['nodes']}")
print(f"    Operation: {functor['preserved']['operation']}")
for e in functor['preserved']['edges'][:4]:
    print(f"    Edge: {e['parent']}→{e['child']}: {e['tau_equals_p']}")
    print(f"                                {e['rho_lost']}")

print("\n  LOST in BN (Ring → BN forgets):")
for key, desc in functor['lost'].items():
    print(f"    {key}: {desc}")

print("\n  GAINED in BN (Ring → BN adds):")
for key, desc in functor['gained'].items():
    print(f"    {key}: {desc}")

FORGETFUL FUNCTOR: Ring → BN

  PRESERVED (in both BSS and BN):
    Nodes: ['A', 'B', 'C', 'D']
    Operation: intersect (used in CPT computation)
    Edge: B→A: τ=0.6857 = P(A|B)=0.6857
                                ρ=0.3429 (not in BN)
    Edge: B→C: τ=0.4857 = P(A|B)=0.4857
                                ρ=0.2571 (not in BN)
    Edge: A→C: τ=0.2000 = P(A|B)=0.2000
                                ρ=0.5714 (not in BN)

  LOST in BN (Ring → BN forgets):
    xor: Ring addition (XOR) has no BN analogue
    rho: BSS exclusion metric ρ is not represented
    complement: BN works with positive probabilities only
    bridge_law: A ∪ B = (A △ B) △ (A ∩ B) is algebraic, not probabilistic

  GAINED in BN (Ring → BN adds):
    d_separation: Conditional independence testing
    markov_blanket: Minimal conditioning sets
    belief_propagation: Evidence-driven inference
    bayes_theorem: P(A|B) = P(B|A)·P(A)/P(B)


### 12. What the Ring Keeps That Both Lose

Neither BSS nor BN preserves the **XOR operation** (ring addition).
XOR represents the *symmetric difference* — what is in A or B but
not in both. Neither a partial order nor a probability measure
naturally captures this.

The ring also provides the **bridge law**:

$$A \cup B = (A \triangle B) \triangle (A \cap B)$$

This is an algebraic identity that has no BSS or BN interpretation.
It belongs purely to the ring.

In [17]:
# Demonstrate what ONLY the ring can do
A, B = docs['A'], docs['B']

# XOR: symmetric difference — unique to the ring
xor_result = A.xor(B)
print("RING-ONLY OPERATIONS")
print("=" * 50)
print(f"  A ⊕ B (XOR): |A⊕B| ≈ {xor_result.cardinality():.0f}")
print(f"  BSS interpretation of XOR: ??? (none — τ and ρ don't cover this)")
print(f"  BN interpretation of XOR:  ??? (P(A⊕B) is not standard probability)")

# Self-inverse: A ⊕ A = ∅
self_xor = A.xor(A)
print(f"\n  A ⊕ A = ∅: |A⊕A| ≈ {self_xor.cardinality():.0f}  (characteristic 2)")
print(f"  This ring identity has no BSS or BN analogue.")

# Bridge law at the HLLSet level
union_direct = A.union(B)
bridge_result = A.xor(B).xor(A.intersect(B))  # (A △ B) △ (A ∩ B)
print(f"\n  Bridge law: |A ∪ B| ≈ {union_direct.cardinality():.0f}")
print(f"  (A △ B) △ (A ∩ B)  ≈ {bridge_result.cardinality():.0f}")
print(f"  Match: {'✓' if abs(union_direct.cardinality() - bridge_result.cardinality()) < 2 else '≈'}")

RING-ONLY OPERATIONS
  A ⊕ B (XOR): |A⊕B| ≈ 22
  BSS interpretation of XOR: ??? (none — τ and ρ don't cover this)
  BN interpretation of XOR:  ??? (P(A⊕B) is not standard probability)

  A ⊕ A = ∅: |A⊕A| ≈ 0  (characteristic 2)
  This ring identity has no BSS or BN analogue.

  Bridge law: |A ∪ B| ≈ 46
  (A △ B) △ (A ∩ B)  ≈ 46
  Match: ✓


---
## Part VI — The Love-Hate Triangle in Practice

### 13. Triangle Analysis

The `triangle_analysis()` method provides a structured view of
all three vertices and their relationships.

In [18]:
analysis = bn.triangle_analysis()

print("THE LOVE-HATE TRIANGLE")
print("=" * 60)

print("\n[1] RING (ground truth):")
for op, desc in analysis['ring'].items():
    print(f"    {op}: {desc}")

print(f"\n[2] BSS LATTICE (structural view):")
print(f"    Edges: {analysis['bss']['edges']}")
print(f"    Interpretation: {analysis['bss']['interpretation']}")
print(f"    Extra: {analysis['bss']['extra_info']}")
print(f"    Reveals: {analysis['bss']['reveals']}")

print(f"\n[3] BAYESIAN NETWORK (epistemic view):")
print(f"    Edges: {analysis['bn']['edges']}")
print(f"    Interpretation: {analysis['bn']['interpretation']}")
print(f"    Extra: {analysis['bn']['extra_info']}")
print(f"    Reveals: {analysis['bn']['reveals']}")

print(f"\n[♥] AGREEMENT:")
print(f"    Formula: {analysis['agreement']['formula']}")
print(f"    Graph isomorphic: {analysis['agreement']['graph_isomorphic']}")

print(f"\n[✗] DIVERGENCE:")
print(f"    BSS only: {analysis['divergence']['bss_only']}")
print(f"    BN only:  {analysis['divergence']['bn_only']}")
print(f"    Philosophical: {analysis['divergence']['philosophical']}")

THE LOVE-HATE TRIANGLE

[1] RING (ground truth):
    union: ∪ (lattice join, OR)
    intersect: ∩ (ring mult, AND)
    xor: ⊕ (ring add, XOR)
    diff: \ (set difference)
    complement: ¬ (bitwise NOT)
    bridge_law: A ∪ B = (A △ B) △ (A ∩ B)

[2] BSS LATTICE (structural view):
    Edges: 3
    Interpretation: Each edge weight τ = "inclusion strength"
    Extra: ρ (exclusion) — no BN analogue
    Reveals: Partial order, Hasse diagram, levels, antichains

[3] BAYESIAN NETWORK (epistemic view):
    Edges: 3
    Interpretation: Each edge weight P(A|B) = "conditional probability"
    Extra: Bayes theorem, d-separation — no BSS analogue
    Reveals: Independence structure, Markov blankets, causal ordering

[♥] AGREEMENT:
    Formula: τ(A→B) = |A∩B|/|B| = P(A|B)
    Graph isomorphic: True

[✗] DIVERGENCE:
    BSS only: ρ (exclusion metric), morphism composition law τ+ρ≤1
    BN only:  d-separation, Markov blanket, belief propagation, Bayes theorem
    Philosophical: BSS asks "who contains 

### 14. A Temporal BN from the Lattice

The temporal lattice provides a CAUSAL structure: earlier states
cause later states. This maps naturally to a BN with temporal edges.

In [19]:
from core.hll_lattice import HLLLattice

# Build a temporal lattice
lat = HLLLattice(p_bits=P_BITS)
lat.append([docs['A']], timestamp=1.0)
lat.append([docs['B']], timestamp=2.0)
lat.append([docs['C']], timestamp=3.0)
lat.append([docs['D']], timestamp=4.0)

# Construct temporal BN
temporal_bn = HLLBayesNet.from_lattice(lat, timestamps=[1.0, 2.0, 3.0, 4.0])

print("TEMPORAL BAYESIAN NETWORK")
print("=" * 50)
print(f"  Nodes: {temporal_bn.num_nodes}")
print(f"  Edges: {temporal_bn.num_edges}")
print()

for nid in temporal_bn.node_ids:
    entries = temporal_bn.get_cpt(nid)
    for e in entries:
        print(f"  {e.parent_id} → {e.child_id}: "
              f"P(later|earlier)={e.probability:.4f} "
              f"[τ={e.bss_tau:.4f}]")

print("\n  Interpretation: P(U(t+1)|U(t)) measures how much")
print("  the earlier state 'predicts' the later one.")
print("  High P → the universe is accumulating (earlier ⊂ later).")
print("  Low P → the universe is diverging (new content dominates).")

TEMPORAL BAYESIAN NETWORK
  Nodes: 4
  Edges: 3

  t=1.0 → t=2.0: P(later|earlier)=1.0000 [τ=1.0000]
  t=2.0 → t=3.0: P(later|earlier)=1.0000 [τ=1.0000]
  t=3.0 → t=4.0: P(later|earlier)=1.0000 [τ=1.0000]

  Interpretation: P(U(t+1)|U(t)) measures how much
  the earlier state 'predicts' the later one.
  High P → the universe is accumulating (earlier ⊂ later).
  Low P → the universe is diverging (new content dominates).


---
## Part VII — Are BSS and BN Truly Isomorphic Representations of the Ring?

### 15. The Precise Answer

Let $\mathcal{R}$ be the HLLSet Boolean ring. Define:

- $\Phi_B: \mathcal{R} \to \mathbf{Graph}_B$ — the BSS representation
- $\Phi_N: \mathcal{R} \to \mathbf{Graph}_N$ — the BN representation

**Theorem** (informal): $\Phi_B$ and $\Phi_N$ are graph-isomorphic
via the identity map on nodes and the equality $\tau = P(A|B)$ on edges.

**However**, they are NOT isomorphic as *enriched structures*:

- $\mathbf{Graph}_B$ is enriched with $(\tau, \rho)$ pairs, morphism
  composition, and partial order extraction.
- $\mathbf{Graph}_N$ is enriched with Bayes' theorem, d-separation,
  Markov blankets, and belief propagation.

The relationship is best described by a **forgetful-free adjunction**:

```
    Ring ──Φ_B──→ Graph_B ──forget ρ──→ Weighted Graph ←──forget ⊥──← Graph_N ←──Φ_N── Ring
```

Both functors $\Phi_B$ and $\Phi_N$ factor through the same intermediate
**weighted directed graph** (nodes + edges with weight $|A \cap B|/|B|$).
They differ only in what *additional* structure they build on top of
that shared skeleton.

### 16. The Synthesis

| Vertex | Asks | Answers |
|--------|------|--------|
| **Ring** | What operations are valid? | XOR, AND, OR, complement, bridge law |
| **BSS** | Who contains whom, and how much? | ⊆ order, Hasse diagram, levels, morphisms |
| **BN** | What predicts what, and how strongly? | Independence, Markov blankets, belief updates |

A complete understanding of an HLLSet system requires all three:

1. The **ring** gives you the algebraic toolkit
2. The **BSS lattice** gives you the structural map
3. The **Bayesian network** gives you the inference engine

They are three facets of one crystal — the HLLSet Boolean algebra.

---
## Summary

### What You Learned

1. **The identity** $\tau(A \to B) = P(A|B)$ — BSS and BN share the same formula
2. **Graph isomorphism** — verified edge-by-edge that BSS ≅ BN as weighted graphs
3. **Algebraic non-isomorphism** — different enrichment (ρ vs d-separation)
4. **BN toolbox**: conditional independence, mutual information, belief propagation, Markov blankets
5. **Structure learning = lattice reconstruction** — same algorithm, different names
6. **Forgetful functor** — what the ring keeps that both representations lose (XOR, bridge law)
7. **Temporal BN** — causal structure from the W lattice

### API Reference

| Function / Class | Purpose |
|-----------------|--------|
| `HLLBayesNet` | Bayesian Network over HLLSets |
| `.from_hllsets(docs)` | Structure learning (= lattice reconstruction) |
| `.from_bss_graph(g)` | Isomorphism constructor from BSS lattice |
| `.from_lattice(lat, ts)` | Temporal causal BN from W lattice |
| `.conditional_independence(a, b, given)` | d-separation via set operations |
| `.markov_blanket(node)` | Minimal conditioning set |
| `.belief_propagation(evidence)` | Update beliefs given observations |
| `.isomorphism_witness(bss_graph)` | Formal isomorphism proof |
| `.triangle_analysis()` | Full love-hate triangle report |
| `hllset_mutual_information(a, b, U)` | $I(A; B)$ in bits |
| `conditional_mutual_information(a, b, c, U)` | $I(A; B | C)$ |
| `ring_to_bn_functor(docs)` | Explicit functor documentation |

### The Complete Triangle

$$\boxed{\tau(A \to B) \;=\; P(A \mid B) \;=\; \frac{|A \cap B|}{|B|}}$$

- **Ring**: the ground truth ($\cap$, $\cup$, $\oplus$, $\setminus$)
- **BSS Lattice**: the order ($\subseteq$, Hasse, levels) — *who contains whom*
- **Bayesian Network**: the measure ($P$, $\perp$, $I$) — *what predicts what*

Three representations. One ring. $\square$